# M5.S2 — Advanced MPI: communicate correctly, then efficiently
**IE University | High Performance Computing | Bachelor Computer Science & AI | 2026–27**

**Learning objectives:** (1) explain why communication can limit scaling, (2) distinguish non-blocking point-to-point and collective operations, (3) use one-sided RMA with explicit synchronization, (4) reuse persistent requests, (5) map neighbors with MPI Cartesian topologies, and (6) recognise when dynamic process spawning is not available.

**Pedagogy:** Real problem → Prediction → Code → Run → Result → Explanation → Student experiment → Takeaway.

**Important:** This notebook has **no pre-populated cluster outputs**. Code-cell execution results are real only when this notebook actually runs. Until a SciTech allocation executes, label MPI measurements **NOT RUN**. The 2026–27 official syllabus was not available when this notebook was prepared; topic mapping is provisionally based on the uploaded M5.S2 slides, legacy notebook and `hpc_course` conventions.

**Runtime budget:** ~80 minutes. Core experiments 1–5, optional experiment 6. No GPU required. The next session treats MPI + OpenMP + GPU hybrid computing.

## 0. Open in IE JupyterLab and understand where code runs
1. Open the university JupyterHub/JupyterLab service using your own university account.
2. Choose **File → Open from URL…** and paste the raw GitHub notebook link in the session README.
3. Choose the **Python 3** kernel and save a personal copy in your own workspace.
4. Jupyter may be on a **login/shared host**, not on an allocated compute node. Heavy MPI examples must run via **Slurm**, never as uncontrolled background `mpirun` processes on the login node.
5. This notebook writes small C programs and a short `sbatch` script into your own `m5s2_demo_files` directory. It does not install packages, request GPUs or access shared datasets.
6. Set `RUN_CLUSTER = True` only after confirming SciTech access and the desired partition. The notebook submits **one short 4-task CPU job** and checks its output; optional spawn is disabled. See [cluster access guide](https://github.com/OscarDiez/hpc_course/blob/main/cluster/README.md).

Typical partitions mentioned in the repository: `cpu`, `interactive`, `gpu`; available partitions, node counts and MPI modules **must be confirmed** with `sinfo` and `module avail`. Existing M2.S3 notes identify previous inter-node MPI runtime inconsistencies; all standard demos use **one node**.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, time
from IPython.display import display, Markdown

WORKDIR = Path.cwd() / "m5s2_demo_files"  # Personal, writable Jupyter directory
RUN_CLUSTER = False     # Explicitly change to True after reviewing the Slurm job
CPU_PARTITION = "cpu"   # Verify with: sinfo
MODULE_NAME = "foss/2023b"  # If unavailable, set "" and rely on an existing mpicc in PATH
RUN_SPAWN_EXTENSION = False  # Optional, requires explicit MPI/Slurm support verification
WORKDIR.mkdir(parents=True, exist_ok=True)
print("Workbook directory:", WORKDIR.resolve())
for label,cmd in [("C compiler","gcc"),("MPI compiler","mpicc"),("Slurm submit","sbatch"),("Slurm launcher","srun")]:
    print(f"{label:18}: {shutil.which(cmd) or 'NOT FOUND (might exist inside the compute job)'}")
print("CURRENT JOB:", os.environ.get("SLURM_JOB_ID","not in an allocation"))
print("RUN_CLUSTER:",RUN_CLUSTER,"(no job is submitted if False)")


## 0a. Check the logic without MPI (safe on any Python kernel)
**Real problem:** When ranks contribute 1, 2, 3, 4, what should a collective sum return? What should rank 1 receive when rank 0 writes 42?

**Predict:** Write the two answers first. This Python reference is a **real local computation**, but it does **not validate any MPI runtime or SciTech performance**.

In [ ]:
from math import prod
expected_reduce=sum(range(1,5))
reference_halo={0:100,1:101}
assert expected_reduce==10
assert reference_halo[0]==100 and reference_halo[1]==101
assert 42==int("42")
print("PASS (Python reference only): 4-rank reduction =",expected_reduce,
      "| Rank 1 RMA target =",42,
      "| two rank payloads =",reference_halo)


## 1. Prepare actual MPI C sources
Each program below is a **small real C/MPI application**, not pseudo-code. The next cell creates source files. The reproducible Slurm job compiles them on the allocated node with `mpicc`, runs numerical correctness checks and prints structured `RESULT experiment=... status=PASS/FAIL` lines.

Inspect the C code before execution. You can rerun this cell safely; it only updates files inside your own demo directory.

In [ ]:
PROGRAMS = {
    "overlap": "#include <mpi.h>\n#include <stdio.h>\n#include <stdlib.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);\n  int rank,size; MPI_Comm_rank(MPI_COMM_WORLD,&rank); MPI_Comm_size(MPI_COMM_WORLD,&size);\n  if(size!=2){if(rank==0)puts(\"RESULT experiment=overlap status=FAIL reason=requires_two_ranks\"); MPI_Finalize();return 1;}\n  int peer=1-rank,send=100+rank,recv=-1,ok;\n  MPI_Request request[2];\n  MPI_Barrier(MPI_COMM_WORLD);\n  double start=MPI_Wtime();\n  MPI_Irecv(&recv,1,MPI_INT,peer,31,MPI_COMM_WORLD,&request[0]);\n  MPI_Isend(&send,1,MPI_INT,peer,31,MPI_COMM_WORLD,&request[1]);\n  volatile double independent=1.0;\n  for(int i=0;i<400000;i++) independent+=1.0/(i+1.0);\n  MPI_Waitall(2,request,MPI_STATUSES_IGNORE);\n  double elapsed=MPI_Wtime()-start;\n  ok=(recv==100+peer && independent>1.0);\n  int all_ok; MPI_Allreduce(&ok,&all_ok,1,MPI_INT,MPI_MIN,MPI_COMM_WORLD);\n  if(rank==0) printf(\"RESULT experiment=overlap status=%s received=%d elapsed_s=%.6f note=overlap_not_guaranteed\\n\",all_ok?\"PASS\":\"FAIL\",recv,elapsed);\n  MPI_Finalize();return all_ok?0:2;\n}",
    "rma": "#include <mpi.h>\n#include <stdio.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);\n  int rank,size; MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&size);\n  if(size!=2){if(rank==0)puts(\"RESULT experiment=rma status=FAIL reason=requires_two_ranks\");MPI_Finalize();return 1;}\n  int inbox=-1,value=42; MPI_Win window;\n  MPI_Win_create(&inbox,sizeof(inbox),sizeof(int),MPI_INFO_NULL,MPI_COMM_WORLD,&window);\n  MPI_Win_fence(0,window);           /* collective: begin access/exposure epoch */\n  if(rank==0) MPI_Put(&value,1,MPI_INT,1,0,1,MPI_INT,window);\n  MPI_Win_fence(0,window);           /* collective: complete transfer before reading */\n  int ok=(rank!=1 || inbox==42),all_ok=0;\n  MPI_Allreduce(&ok,&all_ok,1,MPI_INT,MPI_MIN,MPI_COMM_WORLD);\n  if(rank==0) printf(\"RESULT experiment=rma status=%s expected_target=42\\n\",all_ok?\"PASS\":\"FAIL\");\n  MPI_Win_free(&window);MPI_Finalize();return all_ok?0:2;\n}",
    "persistent": "#include <mpi.h>\n#include <stdio.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);\n  int rank,size;MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&size);\n  if(size!=2){if(rank==0)puts(\"RESULT experiment=persistent status=FAIL reason=requires_two_ranks\");MPI_Finalize();return 1;}\n  int peer=1-rank,send=0,recv=-1,ok=1; MPI_Request request[2];\n  MPI_Recv_init(&recv,1,MPI_INT,peer,32,MPI_COMM_WORLD,&request[0]);\n  MPI_Send_init(&send,1,MPI_INT,peer,32,MPI_COMM_WORLD,&request[1]);\n  for(int step=0;step<4;step++){\n    send=rank*100+step;recv=-1;\n    MPI_Startall(2,request);\n    MPI_Waitall(2,request,MPI_STATUSES_IGNORE);\n    if(recv!=peer*100+step) ok=0;\n  }\n  MPI_Request_free(&request[0]);MPI_Request_free(&request[1]);\n  int all_ok;MPI_Allreduce(&ok,&all_ok,1,MPI_INT,MPI_MIN,MPI_COMM_WORLD);\n  if(rank==0)printf(\"RESULT experiment=persistent status=%s iterations=4 reusable_requests=2\\n\",all_ok?\"PASS\":\"FAIL\");\n  MPI_Finalize();return all_ok?0:2;\n}",
    "ireduce": "#include <mpi.h>\n#include <stdio.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);\n  int rank,size;MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&size);\n  int local=rank+1,total=-1; MPI_Request request;\n  MPI_Iallreduce(&local,&total,1,MPI_INT,MPI_SUM,MPI_COMM_WORLD,&request);\n  volatile double independent=0;\n  for(int i=0;i<200000;i++) independent+=1.0/(i+1.0);\n  MPI_Wait(&request,MPI_STATUS_IGNORE);\n  int expected=size*(size+1)/2;\n  int ok=(total==expected && independent>0);\n  int all_ok;MPI_Allreduce(&ok,&all_ok,1,MPI_INT,MPI_MIN,MPI_COMM_WORLD);\n  if(rank==0)printf(\"RESULT experiment=ireduce status=%s ranks=%d sum=%d expected=%d\\n\",all_ok?\"PASS\":\"FAIL\",size,total,expected);\n  MPI_Finalize();return all_ok?0:2;\n}",
    "topology": "#include <mpi.h>\n#include <stdio.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);\n  int rank,size;MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&size);\n  if(size!=4){if(rank==0)puts(\"RESULT experiment=topology status=FAIL reason=requires_four_ranks\");MPI_Finalize();return 1;}\n  int dims[2]={0,0},periods[2]={0,0};MPI_Dims_create(size,2,dims);\n  MPI_Comm grid;MPI_Cart_create(MPI_COMM_WORLD,2,dims,periods,0,&grid);\n  int grid_rank,coords[2];MPI_Comm_rank(grid,&grid_rank);MPI_Cart_coords(grid,grid_rank,2,coords);\n  int left,right,received=-1;MPI_Cart_shift(grid,1,1,&left,&right);\n  MPI_Sendrecv(&grid_rank,1,MPI_INT,right,33,&received,1,MPI_INT,left,33,grid,MPI_STATUS_IGNORE);\n  int ok=(dims[0]*dims[1]==size) && (left==MPI_PROC_NULL || received==left);\n  int all_ok;MPI_Allreduce(&ok,&all_ok,1,MPI_INT,MPI_MIN,grid);\n  printf(\"OBSERVE experiment=topology rank=%d row=%d col=%d left=%d right=%d received=%d\\n\",\n         grid_rank,coords[0],coords[1],left,right,received);\n  if(grid_rank==0)printf(\"RESULT experiment=topology status=%s grid=%dx%d\\n\",all_ok?\"PASS\":\"FAIL\",dims[0],dims[1]);\n  MPI_Comm_free(&grid);MPI_Finalize();return all_ok?0:2;\n}",
    "spawn_parent": "#include <mpi.h>\n#include <stdio.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);\n  MPI_Comm_set_errhandler(MPI_COMM_SELF,MPI_ERRORS_RETURN);\n  MPI_Comm children=MPI_COMM_NULL;\n  int error_codes[2]={0,0};\n  int rc=MPI_Comm_spawn(\"./spawn_child\",MPI_ARGV_NULL,2,MPI_INFO_NULL,0,MPI_COMM_SELF,&children,error_codes);\n  if(rc!=MPI_SUCCESS || children==MPI_COMM_NULL){\n    puts(\"RESULT experiment=spawn status=SKIPPED reason=mpi_runtime_does_not_support_spawn\");\n    MPI_Finalize();return 0;\n  }\n  MPI_Comm_disconnect(&children);\n  puts(\"RESULT experiment=spawn status=PASS children=2\");\n  MPI_Finalize();return 0;\n}",
    "spawn_child": "#include <mpi.h>\nint main(int argc,char **argv){\n  MPI_Init(&argc,&argv);MPI_Comm parent;\n  MPI_Comm_get_parent(&parent);\n  if(parent!=MPI_COMM_NULL)MPI_Comm_disconnect(&parent);\n  MPI_Finalize();return 0;\n}",
}
for name, source in PROGRAMS.items():
    p = WORKDIR / (name + ".c")
    p.write_text(source + "\n", encoding="utf-8")
print("Prepared",len(PROGRAMS),"real C/MPI source files in",WORKDIR)
print("File examples:",", ".join(sorted(f.name for f in WORKDIR.glob("*.c"))))


### Inspect a program before predicting its result
This prints the **actual source code** that the job will compile. Change `SOURCE_TO_INSPECT` to `rma`, `persistent`, `ireduce`, `topology` or `spawn_parent` to investigate the matching experiment.

In [ ]:
SOURCE_TO_INSPECT = "overlap"
print((WORKDIR / (SOURCE_TO_INSPECT+".c")).read_text(encoding="utf-8"))


## 2. Submit one short, resource-bounded CPU batch job
The job requests **1 node, 4 MPI tasks, 1 CPU per task, 1 GiB and a 5-minute limit**. It runs the core programs, each in its own `srun` step, and records both correctness results and Slurm failures. Spawning children is not part of the default run.

**Safety:** `RUN_CLUSTER=False` means **NOT RUN**, not simulated success. `RUN_CLUSTER=True` submits at most one job for the current source/configuration fingerprint; subsequent runs reuse its job ID until you explicitly reset or change the configuration. Long queue waits are not mistaken for failures.

**SciTech note:** If the cluster modules do not provide `mpicc` inside the job, output will report `SKIPPED/FAIL` with an explanation rather than pretending it ran. `module load` availability varies by installation.

In [ ]:
import shlex

def make_script():
    module_line = (f"if command -v module >/dev/null 2>&1; then module load {shlex.quote(MODULE_NAME)} || true; fi"
                   if MODULE_NAME.strip() else ": # No module requested")
    lines = [
       "#!/usr/bin/env bash", "#SBATCH --job-name=m5s2_mpi", "#SBATCH --nodes=1",
       "#SBATCH --ntasks=4", "#SBATCH --cpus-per-task=1",
       "#SBATCH --time=00:05:00", "#SBATCH --mem=1G",
       "set -u", "set -o pipefail",
       "cd "+shlex.quote(str(WORKDIR.resolve())),
       module_line,
       "if ! command -v mpicc >/dev/null 2>&1; then echo 'JOB_STATUS=SKIPPED missing_mpicc'; exit 10; fi",
       "if ! command -v srun >/dev/null 2>&1; then echo 'JOB_STATUS=SKIPPED missing_srun'; exit 10; fi",
       "failed=0",
       "for prog in overlap rma persistent ireduce topology spawn_parent spawn_child; do",
       "  if ! mpicc -std=c11 -O2 -Wall -Wextra -o \"$prog\" \"$prog.c\"; then",
       "    echo \"COMPILE program=$prog status=FAIL\"; failed=1",
       "  else echo \"COMPILE program=$prog status=PASS\"; fi",
       "done",
       "run_demo() {",
       " name=$1; ranks=$2",
       " if [ ! -x \"./$name\" ]; then echo \"RUN experiment=$name status=SKIPPED reason=compile_failed\"; failed=1; return; fi",
       " echo \"RUN_START experiment=$name ranks=$ranks\"",
       " if srun --nodes=1 --ntasks=\"$ranks\" \"./$name\"; then",
       "   echo \"RUN experiment=$name status=COMPLETED\"",
       " else echo \"RUN experiment=$name status=FAIL\"; failed=1; fi",
       "}",
       "run_demo overlap 2", "run_demo rma 2", "run_demo persistent 2",
       "run_demo ireduce 4", "run_demo topology 4",
    ]
    if RUN_SPAWN_EXTENSION:
        lines += ['echo "WARNING: optional MPI_Comm_spawn requested; cluster support must be independently confirmed"',
                  'run_demo spawn_parent 1']
    else:
        lines += ['echo "RUN experiment=spawn status=SKIPPED reason=optional_spawn_disabled"']
    lines += ['echo "JOB_STATUS_DONE failed=$failed"', 'exit "$failed"']
    return "\n".join(lines)+"\n"

SCRIPT = WORKDIR / "m5s2_submit.sbatch"
SCRIPT.write_text(make_script(), encoding="utf-8")
fingerprint = hashlib.sha256((make_script()+"".join(PROGRAMS.values())).encode()).hexdigest()[:12]
print("Script:", SCRIPT, "| config fingerprint:",fingerprint)
print("Slurm request: 1 node, 4 tasks, CPU partition,",CPU_PARTITION,", 5 minutes, 1 GiB")
print("Core experiments: overlap, RMA, persistent, Iallreduce, Cartesian topology")
print("Optional spawn enabled:", RUN_SPAWN_EXTENSION)


In [ ]:
STATE_FILE = WORKDIR / "m5s2_job_state.json"
JOB_ID = None
if not RUN_CLUSTER:
    print("NOT RUN: Set RUN_CLUSTER=True only after checking SciTech access and partition.")
elif not shutil.which("sbatch"):
    print("SKIPPED: sbatch not available to this Jupyter kernel. Use SciTech JupyterHub/Slurm.")
else:
    previous = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    if previous.get("fingerprint")==fingerprint and previous.get("job_id"):
        JOB_ID = str(previous["job_id"])
        print("REUSED previous Slurm submission",JOB_ID,"(no duplicate job created)")
    else:
        cmd=["sbatch","--parsable","-p",CPU_PARTITION,
             "--output",str(WORKDIR / f"m5s2_{fingerprint}_%j.out"),
             "--error",str(WORKDIR / f"m5s2_{fingerprint}_%j.err"),
             str(SCRIPT)]
        result=subprocess.run(cmd,capture_output=True,text=True,timeout=25)
        if result.returncode:
            print("FAIL: sbatch submission failed:",result.stderr.strip()[:1000])
        else:
            JOB_ID=result.stdout.strip().split(";")[0]
            if not JOB_ID.isdigit():
                print("FAIL: unexpected sbatch response",result.stdout[:300])
                JOB_ID=None
            else:
                STATE_FILE.write_text(json.dumps({"fingerprint":fingerprint,"job_id":JOB_ID},indent=2))
                print("SUBMITTED: Slurm job",JOB_ID)
if JOB_ID:
    print("Check job: squeue -j",JOB_ID)
    print("Outputs will be in",WORKDIR,"and may take time to arrive.")


In [ ]:
def load_job_output(wait_seconds=45):
    if not JOB_ID:
        print("NOT RUN: there is no submitted/reused Slurm job in this session.")
        return ""
    output=WORKDIR / f"m5s2_{fingerprint}_{JOB_ID}.out"
    error=WORKDIR / f"m5s2_{fingerprint}_{JOB_ID}.err"
    until=time.monotonic()+wait_seconds
    while time.monotonic()<until:
        if output.exists() and "JOB_STATUS_DONE" in output.read_text(errors="replace"):
            break
        time.sleep(2)
    if not output.exists():
        print("NOT RUN/QUEUED: output not ready. Check squeue and rerun this cell later.")
        return ""
    out=output.read_text(errors="replace")
    print("REAL Slurm stdout file:",output)
    print(out[-14000:])
    if error.exists() and error.stat().st_size:
        print("REAL Slurm stderr (last 2000 chars):",error.read_text(errors="replace")[-2000:])
    if "JOB_STATUS_DONE" not in out:
        print("INCOMPLETE: output exists, but job completion is unconfirmed.")
    return out

M5S2_OUTPUT = load_job_output()


In [ ]:
def experiment_status(name, output=M5S2_OUTPUT):
    if not output:
        status="NOT RUN"
    else:
        lines=[x for x in output.splitlines() if x.startswith(f"RESULT experiment={name} ")]
        core = [x for x in output.splitlines() if x.startswith(f"RUN experiment={name} ")]
        if core and "status=SKIPPED" in core[-1]:
            status="SKIPPED"
        elif core and "status=FAIL" in core[-1]:
            status="FAIL"
        elif lines and "status=PASS" in lines[-1] and core and "status=COMPLETED" in core[-1]:
            status="PASS"
        elif lines and "status=SKIPPED" in lines[-1]:
            status="SKIPPED"
        elif lines and "status=FAIL" in lines[-1]:
            status="FAIL"
        else:
            status="NOT RUN"
    print(f"{name:12} {status}")
    return status

M5S2_RESULTS={name:experiment_status(name) for name in ("overlap","rma","persistent","ireduce","topology","spawn")}


## 3. Experiment 1 — Non-blocking halo communication
**Real problem:** A temperature-grid rank sends boundary data but also has useful interior computation.

**[PREDICT]** Will `MPI_Isend` mean the transfer is already complete? Predict the value received by rank 0.

**Code:** Read `PROGRAMS["overlap"]`. MPI_Irecv + MPI_Isend; independent CPU work; MPI_Waitall; check partner's payload.

**[DEMO]** Execute the one controlled Slurm job above. The next cell shows its **actual** pass/fail state, or **NOT RUN/SKIPPED** if no SciTech evidence exists.

In [ ]:
print("Experiment 1: Non-blocking halo communication")
experiment_status("overlap")
if M5S2_OUTPUT:
    print("\n".join(x for x in M5S2_OUTPUT.splitlines() if "experiment=overlap " in x)[-1800:])


**[EXPLAIN]** Non-blocking lets work proceed before waiting; progress and speedup are implementation-dependent.

**[ACTIVITY: change ONE parameter]** Changing `independent` work-loop iterations in `overlap.c` from 400000 to 100000. Rebuild the source and submit a fresh fingerprint, compare actual elapsed time.

**Write your observation:** I predicted ___. I changed ___. My measured result was ___ (or NOT RUN). I now think ___.

**Takeaway:** Non-blocking lets work proceed before waiting; progress and speedup are implementation-dependent.

## 4. Experiment 2 — One-sided remote memory access (RMA)
**Real problem:** Rank 0 must update a value owned by rank 1 without a matching receive call.

**[PREDICT]** Will rank 1 necessarily see 42 before or after the closing fence?

**Code:** Read `PROGRAMS["rma"]`. MPI_Win_create / MPI_Put / two collective MPI_Win_fence calls; check target after fence.

**[DEMO]** Execute the one controlled Slurm job above. The next cell shows its **actual** pass/fail state, or **NOT RUN/SKIPPED** if no SciTech evidence exists.

In [ ]:
print("Experiment 2: One-sided remote memory access (RMA)")
experiment_status("rma")
if M5S2_OUTPUT:
    print("\n".join(x for x in M5S2_OUTPUT.splitlines() if "experiment=rma " in x)[-1800:])


**[EXPLAIN]** RMA avoids a matching receive but does **not** remove synchronization requirements.

**[ACTIVITY: change ONE parameter]** Change the constant `value=42` to `value=17` and update expected target; does the correct run still PASS?

**Write your observation:** I predicted ___. I changed ___. My measured result was ___ (or NOT RUN). I now think ___.

**Takeaway:** RMA avoids a matching receive but does **not** remove synchronization requirements.

## 5. Experiment 3 — Persistent requests for iterative work
**Real problem:** A solver repeatedly exchanges the same-sized boundary value across four iterations.

**[PREDICT]** What is reused: the content, the buffer address, the partner or the request handle?

**Code:** Read `PROGRAMS["persistent"]`. MPI_Send_init / MPI_Recv_init once, then Startall / Waitall four times; verify each received value.

**[DEMO]** Execute the one controlled Slurm job above. The next cell shows its **actual** pass/fail state, or **NOT RUN/SKIPPED** if no SciTech evidence exists.

In [ ]:
print("Experiment 3: Persistent requests for iterative work")
experiment_status("persistent")
if M5S2_OUTPUT:
    print("\n".join(x for x in M5S2_OUTPUT.splitlines() if "experiment=persistent " in x)[-1800:])


**[EXPLAIN]** Persistent requests reuse communication setup, **not** previously transmitted data.

**[ACTIVITY: change ONE parameter]** Change the number of steps `step<4` to `step<6`; update reported iteration count and rerun.

**Write your observation:** I predicted ___. I changed ___. My measured result was ___ (or NOT RUN). I now think ___.

**Takeaway:** Persistent requests reuse communication setup, **not** previously transmitted data.

## 6. Experiment 4 — Non-blocking collective reduction
**Real problem:** Four workers each produce a partial result: 1, 2, 3 and 4.

**[PREDICT]** Predict the final sum; can local independent computation run while a reduction request exists?

**Code:** Read `PROGRAMS["ireduce"]`. MPI_Iallreduce, independent work, MPI_Wait; check total 10.

**[DEMO]** Execute the one controlled Slurm job above. The next cell shows its **actual** pass/fail state, or **NOT RUN/SKIPPED** if no SciTech evidence exists.

In [ ]:
print("Experiment 4: Non-blocking collective reduction")
experiment_status("ireduce")
if M5S2_OUTPUT:
    print("\n".join(x for x in M5S2_OUTPUT.splitlines() if "experiment=ireduce " in x)[-1800:])


**[EXPLAIN]** Non-blocking collectives can expose work for overlap, but may not guarantee progress.

**[ACTIVITY: change ONE parameter]** Change rank count in job step from 4 to 2 and predict the sum (3). Reset the fingerprint before re-submitting.

**Write your observation:** I predicted ___. I changed ___. My measured result was ___ (or NOT RUN). I now think ___.

**Takeaway:** Non-blocking collectives can expose work for overlap, but may not guarantee progress.

## 7. Experiment 5 — Cartesian topologies for grid neighbors
**Real problem:** A 2x2 grid exchanges a value with its left neighbor.

**[PREDICT]** Is every rank guaranteed to have a left neighbor? What is MPI_PROC_NULL?

**Code:** Read `PROGRAMS["topology"]`. MPI_Dims_create + MPI_Cart_create + MPI_Cart_shift + MPI_Sendrecv; print each rank's coordinates.

**[DEMO]** Execute the one controlled Slurm job above. The next cell shows its **actual** pass/fail state, or **NOT RUN/SKIPPED** if no SciTech evidence exists.

In [ ]:
print("Experiment 5: Cartesian topologies for grid neighbors")
experiment_status("topology")
if M5S2_OUTPUT:
    print("\n".join(x for x in M5S2_OUTPUT.splitlines() if "experiment=topology " in x)[-1800:])


**[EXPLAIN]** A Cartesian communicator expresses logical neighbors; it does not automatically guarantee physical locality.

**[ACTIVITY: change ONE parameter]** Change nonperiodic grid to periodic `periods[1]=1` and predict the wrap-around neighbor.

**Write your observation:** I predicted ___. I changed ___. My measured result was ___ (or NOT RUN). I now think ___.

**Takeaway:** A Cartesian communicator expresses logical neighbors; it does not automatically guarantee physical locality.

## 8. Experiment 6 — Optional: dynamic processes (optional)
**Real problem:** An adaptive simulation might request extra workers while running.

**[PREDICT]** Can a Slurm job always grow past its fixed allocation using MPI_Comm_spawn?

**Code:** Read `PROGRAMS["spawn"]`. MPI_Comm_spawn with a child program, disabled by default. Execute only if cluster admins confirm supported MPI/Slurm configuration.

**[DEMO]** Execute the one controlled Slurm job above. The next cell shows its **actual** pass/fail state, or **NOT RUN/SKIPPED** if no SciTech evidence exists.

In [ ]:
print("Experiment 6: Optional: dynamic processes")
experiment_status("spawn")
if M5S2_OUTPUT:
    print("\n".join(x for x in M5S2_OUTPUT.splitlines() if "experiment=spawn " in x)[-1800:])


**[EXPLAIN]** MPI spawn is optional and cluster-dependent; ordinary Slurm jobs should assume fixed allocations.

**[ACTIVITY: change ONE parameter]** Ask what happens when the runtime cannot spawn, and why extra processes do not automatically imply extra Slurm resources.

**Write your observation:** I predicted ___. I changed ___. My measured result was ___ (or NOT RUN). I now think ___.

**Takeaway:** MPI spawn is optional and cluster-dependent; ordinary Slurm jobs should assume fixed allocations.

## 9. Compare all experiments and explain failures
In your own words, explain why *correct output* is separate from *fast output*. For example, a non-blocking request is not necessarily progressing while the CPU computes. A topology declaration does not force a particular hardware mapping. If the SciTech MPI stack, module, allocation or process spawn is unavailable, explain the failure rather than claiming a measurement.

In [ ]:
from collections import Counter
print("REAL MPI experiment status report:",dict(Counter(M5S2_RESULTS.values())))
for name,state in M5S2_RESULTS.items():
    print(f"  {name:<12} {state}")
if not M5S2_OUTPUT:
    print("SciTech MPI measurements: NOT RUN. Python reference validation above is separate.")


## 10. Quick reasoning checkpoints (answers below for self-study)
1. Why is calling `MPI_Waitall` immediately after `MPI_Isend` unlikely to help overlap?
2. Does an `MPI_Put` eliminate the need for target memory exposure and synchronization?
3. Which part of a persistent send is reused between iterations?
4. For ranks 0–3 contributing `rank+1`, what should `MPI_Iallreduce(MPI_SUM)` return?
5. Is `MPI_Cart_create` a guarantee that neighboring logical ranks occupy neighboring physical compute nodes?
6. Can `MPI_Comm_spawn` automatically reserve more physical nodes outside a Slurm allocation?

**Expected reasoning:** (1) Waiting immediately forfeits useful independent work; (2) no, an access/exposure epoch is required; (3) a reusable communication request with a fixed buffer and partner definition; (4) 10; (5) no, logical topology and placement are distinct; (6) no, support and scheduling are separate concerns.

## 11. Four essentials from M5.S2 and bridge to M5.S3
- **Overlap:** non-blocking communication separates initiation from completion; verify correctness first.
- **Control:** one-sided communication requires explicit epochs/synchronization.
- **Repeat:** persistent requests can amortize setup for a recurring pattern.
- **Layout and scaling:** collective/topology choices and load balance affect performance; cluster support matters.

**Exit ticket:** You are writing a 2D heat solver on many nodes. Which single MPI technique would you test first, how would you prove its output is correct, and what timing would you measure?

**Next session:** MPI + OpenMP + GPU uses the same communication principles across heterogeneous nodes. Do not confuse MPI rank topology with OpenMP threads or GPU kernels.

**Validation statement:** This notebook itself has no prefilled outputs. Local Python reference checks run on the user's kernel; actual MPI PASS results require a successful SciTech job output.